## Setup and Imports

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

CFG = dict(
    SEED          = 42,
    N_FOLDS       = 2,
    EPOCHS_LSTM   = 4,
    BS_LSTM       = 256,
    LR_LSTM       = 1e-3,
    LSTM_HIDDEN   = 384,
    MARGIN        = 0.5,
    LSTM_EMBED    = 256,
    DATA_DIR      = '/kaggle/input/competitions/smart-mcq-solver-challenge',
    OUTPUT_DIR    = '/kaggle/working',
    ARTIFACT_DIR  = '/kaggle/working/artifacts'
)

os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)
os.makedirs(CFG['ARTIFACT_DIR'], exist_ok=True)
OPTS = ['A', 'B', 'C', 'D', 'E']

In [6]:
import os, sys, json, math, random, time, gc, re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast as _autocast, GradScaler as _GradScaler

from sklearn.metrics import f1_score, accuracy_score


def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(CFG['SEED'])

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

AMP_DTYPE = torch.float16
USE_SCALER = True
print(f"AMP dtype: {AMP_DTYPE} | GradScaler: {USE_SCALER}")

def autocast():
    return _autocast(device_type='cuda', dtype=AMP_DTYPE)

def make_scaler():
    return _GradScaler('cuda')

Device: cpu
AMP dtype: torch.float16 | GradScaler: True


## Load Data

In [8]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
y_idx = np.array([OPTS.index(a) for a in train['answer']])

## Normalize + GroupKFold Assignment

In [9]:
from sklearn.model_selection import GroupKFold

_BOILERPLATE_PAT = re.compile(
    r"pick the best possible answer:?|\bcarefully\b\.?|among the listed options\.?|"
    r"choose the (correct|best) (option|answer|response)\.?|"
    r"select the (best|correct|most appropriate) (option|answer|response)\.?|"
    r"determine the correct option:?|identify the correct statement:?|"
    r"based on the given context\.?|which of the following( statements)?( is true)?",
    flags=re.IGNORECASE
)
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = _BOILERPLATE_PAT.sub(' ', str(text))
    return _WS.sub(' ', s).strip()


for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)


gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.iloc[va_idx, train.columns.get_loc('fold')] = fold_id


assert (train['fold'] >= 0).all() and train.groupby('prompt_n')['fold'].nunique().max() == 1, "Leak detected!"
print("Folds:\n", train['fold'].value_counts().sort_index().to_string())

Folds:
 fold
0    1000
1    1000


## Metrics & Loss 

In [5]:
def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = list(order[i, :3])
        if y_idx[i] in top3:
            total += 1.0 / (top3.index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    pred = scores_2d.argmax(axis=1)
    return dict(map3=map_at_3(scores_2d, y_idx),
                acc=accuracy_score(y_idx, pred),
                f1=f1_score(y_idx, pred, average='macro'))

In [6]:
def pairwise_margin_loss(scores, correct, margin=0.5):

    B = scores.size(0)
    pos = scores[torch.arange(B), correct].unsqueeze(1).expand(-1, 5)
    target = torch.ones_like(scores)
    loss = F.margin_ranking_loss(pos, scores, target, margin=margin, reduction='none')
    mask = torch.ones_like(loss)
    mask[torch.arange(B), correct] = 0
    return (loss * mask).sum() / mask.sum().clamp(min=1)

### Tracking (Weights & Biases)


In [7]:
import os
try:
    import wandb
except ImportError:
    wandb = None

USE_WANDB = True
RUN_VERSION = 3
RUN_NOTES = "lstm_v2, n_folds = 2, epochs = 4"         
WANDB_PROJECT = "24f1000134-t22026"

if USE_WANDB and wandb is not None:
    try:
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("wandb_api"))
        print("W&B connected.")
    except Exception as e:
        print(f"W&B login skipped ({e}); set Kaggle secret 'wandb_api' to enable.")
        USE_WANDB = False
elif wandb is None:
    USE_WANDB = False
    print("wandb not installed; logging disabled.")

def wandb_init(model_tag, fold):
    if not (USE_WANDB and wandb is not None):
        return None
    return wandb.init(
        project=WANDB_PROJECT,
        name=f"{model_tag}_v{RUN_VERSION}_fold{fold}",
        group=f"{model_tag}_v{RUN_VERSION}",    
        job_type="train",
        notes=RUN_NOTES,                               
        tags=[model_tag, f"v{RUN_VERSION}"],
        config={**CFG, "run_version": RUN_VERSION, "run_notes": RUN_NOTES, "model": model_tag, "fold": fold},
        reinit=True,
    )

def wlog(metrics, step=None):
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.log(metrics, step=step)

def wandb_finish():
    if USE_WANDB and wandb is not None and wandb.run is not None:
        wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f1000134 (24f1000134-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B connected.


### Uploading Model to Kagglehub

In [8]:
try:
    import kagglehub
except ImportError:
    kagglehub = None


KAGGLE_USERNAME = "kamal134134"
PUSH_TO_HUB     = True
HUB_FRAMEWORK   = "pyTorch"

HUB_HANDLES = {
    "lstm":    f"{KAGGLE_USERNAME}/genai_project/{HUB_FRAMEWORK}/lstm_1"
}

def push_model(model_tag, folder, notes=""):
    if not PUSH_TO_HUB or kagglehub is None:
        print(f"[hub] upload skipped for '{model_tag}' (PUSH_TO_HUB=False).")
        return
    handle = HUB_HANDLES[model_tag]
    note = notes or f"v{RUN_VERSION}: {RUN_NOTES}"
    print(f"[hub] uploading '{model_tag}' -> {handle}")
    kagglehub.model_upload(handle, folder, version_notes=note)
    print(f"[hub] done. Inference notebooks can now kagglehub.model_download('{handle}').")

## 1. Vocabulary

In [9]:
PAD, UNK = 0, 1
LSTM_MAX_OPT_LEN = 25
LSTM_MAX_Q_LEN   = 80

def build_vocab(texts, min_freq=2, max_size=30000):
    cnt = Counter()
    for t in texts:
        cnt.update(re.findall(r"[a-zA-Z']+|\d+", t.lower()))
    vocab = {'<pad>': PAD, '<unk>': UNK}
    for w, c in cnt.most_common(max_size):
        if c >= min_freq:
            vocab[w] = len(vocab)
    return vocab

def encode_text(text, vocab, max_len):
    toks = re.findall(r"[a-zA-Z']+|\d+", text.lower())[:max_len]
    ids = [vocab.get(t, UNK) for t in toks]
    ids += [PAD] * (max_len - len(ids))
    return np.array(ids, dtype=np.int64)

_all = [train['prompt_n'].tolist(), test['prompt_n'].tolist()]
for o in OPTS:
    _all += [train[f'{o}_n'].tolist(), test[f'{o}_n'].tolist()]
VOCAB = build_vocab([t for grp in _all for t in grp])
print(f"LSTM vocab size: {len(VOCAB)}")

LSTM vocab size: 2983


## 2. Dataset 

In [10]:
class LSTMDataset(Dataset):
    def __init__(self, df, vocab, shuffle_opts=True, has_label=True, seed=0):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.shuffle = shuffle_opts
        self.has_label = has_label
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        q_ids = encode_text(r['prompt_n'], self.vocab, LSTM_MAX_Q_LEN)
        opt_ids = np.stack([encode_text(r[f'{o}_n'], self.vocab, LSTM_MAX_OPT_LEN)
                            for o in OPTS])
        correct = OPTS.index(r['answer']) if self.has_label else -1
        if self.shuffle:
            perm = self.rng.permutation(5)
            opt_ids = opt_ids[perm]
            if self.has_label:
                correct = int(np.where(perm == correct)[0][0])
        return dict(q=torch.from_numpy(q_ids),
                    opts=torch.from_numpy(opt_ids),
                    correct=torch.tensor(correct, dtype=torch.long),
                    id=int(r['id']))

In [11]:
class SiameseBiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=256, hidden=384, dropout=0.3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True,
                            bidirectional=True, num_layers=1)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(4 * 2 * hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, 1),
        )

    def encode(self, ids):
        emb = self.embed(ids)
        out, _ = self.lstm(emb)
        mask = (ids != PAD).unsqueeze(-1).float()
        out = out * mask + (1 - mask) * -1e9
        pooled, _ = out.max(dim=1)
        return self.dropout(pooled)

    def forward(self, q_ids, opt_ids):
        B = q_ids.size(0)
        h_q = self.encode(q_ids)
        h_o = self.encode(opt_ids.view(B * 5, -1)).view(B, 5, -1)
        h_q = h_q.unsqueeze(1).expand(-1, 5, -1)
        joined = torch.cat([h_q, h_o, torch.abs(h_q - h_o), h_q * h_o], dim=-1)
        return self.head(joined).squeeze(-1)

## 4. Cross-validated training



In [12]:
MODEL_TAG  = 'lstm'
UPLOAD_DIR = f"{CFG['ARTIFACT_DIR']}/{MODEL_TAG}"
os.makedirs(UPLOAD_DIR, exist_ok=True)


@torch.no_grad()
def predict_lstm(model, dl):
    model.eval()
    out = []
    for batch in dl:
        q    = batch['q'].to(DEVICE, non_blocking=True)
        opts = batch['opts'].to(DEVICE, non_blocking=True)
        with autocast():
            s = model(q, opts)
        out.append(s.cpu().float().numpy())
    return np.concatenate(out)


def train_lstm_cv(train, test_df, n_folds=CFG['N_FOLDS'], save_dir=UPLOAD_DIR):
    oof     = np.zeros((len(train), 5), dtype=np.float32)
    te_pred = np.zeros((len(test_df),  5), dtype=np.float32)
    y       = np.array([OPTS.index(a) for a in train['answer']])

    te_dl = DataLoader(LSTMDataset(test_df, VOCAB, shuffle_opts=False, has_label=False),
                       batch_size=CFG['BS_LSTM'], shuffle=False, pin_memory=True)

    for fold in range(n_folds):
     
        run = wandb_init(MODEL_TAG, fold)

        va_idx = np.where(train['fold'].values == fold)[0]
        tr_idx = np.where(train['fold'].values != fold)[0]
        print(f'\n[LSTM] === fold {fold} (train={len(tr_idx)}, val={len(va_idx)}) ===', flush=True)

        tr_dl = DataLoader(LSTMDataset(train.iloc[tr_idx], VOCAB, True, seed=fold),
                           batch_size=CFG['BS_LSTM'], shuffle=True, pin_memory=True)
        va_dl = DataLoader(LSTMDataset(train.iloc[va_idx], VOCAB, False),
                           batch_size=CFG['BS_LSTM'], shuffle=False, pin_memory=True)

        model  = SiameseBiLSTM(len(VOCAB), CFG['LSTM_EMBED'], CFG['LSTM_HIDDEN']).to(DEVICE)
        optim  = AdamW(model.parameters(), lr=CFG['LR_LSTM'], weight_decay=1e-5)
        sched  = CosineAnnealingLR(optim, T_max=CFG['EPOCHS_LSTM'] * max(len(tr_dl), 1))
        scaler = make_scaler()

        best_map3, best_va, best_te = -1, None, None

        for epoch in range(CFG['EPOCHS_LSTM']):
            t_epoch = time.time()
            model.train()
            running_loss, seen = 0.0, 0

            for step, batch in enumerate(tr_dl):
                q    = batch['q'].to(DEVICE, non_blocking=True)
                opts = batch['opts'].to(DEVICE, non_blocking=True)
                correct = batch['correct'].to(DEVICE, non_blocking=True)

                optim.zero_grad()
                with autocast():
                    scores = model(q, opts)
                    loss   = pairwise_margin_loss(scores, correct, CFG['MARGIN'])

                if torch.isfinite(loss):
                    scaler.scale(loss).backward()
                else:
                    print(f'    [warn] step {step}: non-finite loss, skipping backward')

                if USE_SCALER: scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optim); scaler.update()
                sched.step()

                running_loss += loss.item() * q.size(0)
                seen += q.size(0)

              
                if step % 25 == 0:
                    print(f'    step {step}/{len(tr_dl)}  '
                          f'loss={loss.item():.4f}  '
                          f'elapsed={time.time()-t_epoch:.0f}s', flush=True)

       
            model.eval()
            va_loss, va_seen = 0.0, 0
            va_scores_all = []
            with torch.no_grad():
                for batch in va_dl:
                    q    = batch['q'].to(DEVICE, non_blocking=True)
                    opts = batch['opts'].to(DEVICE, non_blocking=True)
                    correct = batch['correct'].to(DEVICE, non_blocking=True)
                    with autocast():
                        scores = model(q, opts)
                        v_loss = pairwise_margin_loss(scores, correct, CFG['MARGIN'])
                    va_loss += v_loss.item() * q.size(0)
                    va_seen += q.size(0)
                    va_scores_all.append(scores.cpu().float().numpy())
            va_scores = np.concatenate(va_scores_all)
            m = all_metrics(va_scores, y[va_idx])

            tr_loss_val = running_loss / max(seen, 1)
            va_loss_val = va_loss / max(va_seen, 1)

     
            print(f'  ep{epoch} ({time.time()-t_epoch:.0f}s)  '
                  f'train_loss={tr_loss_val:.4f} '
                  f'val_loss={va_loss_val:.4f} val_map3={m["map3"]:.4f} '
                  f'val_acc={m["acc"]:.4f} val_f1={m["f1"]:.4f}', flush=True)

       
            wlog({
                'lstm/train_loss': tr_loss_val,
                'lstm/val_loss':   va_loss_val,
                'lstm/val_map3':   m['map3'],
                'lstm/val_acc':    m['acc'],
                'lstm/val_f1':     m['f1'],
                'lstm/fold':       fold,
                'lstm/epoch':      epoch,
            })

           
            if m['map3'] > best_map3:
                best_map3, best_va = m['map3'], va_scores
                best_te = predict_lstm(model, te_dl)
                torch.save(model.state_dict(), f"{save_dir}/best_lstm_fold{fold}.pth")
                print(f"    -> new best (val_map3={best_map3:.4f}), checkpoint saved", flush=True)

        wlog({'lstm/best_val_map3': best_map3})
        oof[va_idx] = best_va
        te_pred += best_te / n_folds
        wandb_finish()
        del model; gc.collect(); torch.cuda.empty_cache()

    overall = all_metrics(oof, y)
    print(f'\n[LSTM] OOF MAP@3={overall["map3"]:.4f} '
          f'acc={overall["acc"]:.4f} f1={overall["f1"]:.4f}', flush=True)


    if USE_WANDB and wandb is not None:
        summary_run = wandb.init(
            project=WANDB_PROJECT,
            name=f"{MODEL_TAG}_v{RUN_VERSION}_OOF",
            group=f"{MODEL_TAG}_v{RUN_VERSION}",
            job_type="summary", tags=[MODEL_TAG, f"v{RUN_VERSION}", "oof"],
            reinit=True,
        )
        wandb.log({
            'lstm/oof_map3': overall['map3'],
            'lstm/oof_acc':  overall['acc'],
            'lstm/oof_f1':   overall['f1'],
        })
        wandb.run.summary['oof_map3'] = overall['map3']
        wandb.run.summary['oof_acc']  = overall['acc']
        wandb.run.summary['oof_f1']   = overall['f1']
        wandb.finish()


    np.save(f"{save_dir}/oof_lstm.npy",  oof)
    np.save(f"{save_dir}/test_lstm.npy", te_pred)
    print(f"Saved test_lstm.npy + oof_lstm.npy.")


    push_model(
        MODEL_TAG, save_dir,
        notes=f"v{RUN_VERSION}: {RUN_NOTES} | OOF MAP@3={overall['map3']:.4f} "
              f"acc={overall['acc']:.4f} f1={overall['f1']:.4f}"
    )

    return oof, te_pred

oof_lstm, test_lstm = train_lstm_cv(train, test)
print(f"\nTraining complete. test_lstm shape={test_lstm.shape}, oof_lstm shape={oof_lstm.shape}")

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



[LSTM] === fold 0 (train=1000, val=1000) ===
    step 0/4  loss=0.5055  elapsed=1s
  ep0 (2s)  train_loss=0.4737 val_loss=0.4207 val_map3=0.7258 val_acc=0.5980 val_f1=0.5918
    -> new best (val_map3=0.7258), checkpoint saved
    step 0/4  loss=0.3779  elapsed=0s
  ep1 (1s)  train_loss=0.3269 val_loss=0.2934 val_map3=0.8045 val_acc=0.7160 val_f1=0.7131
    -> new best (val_map3=0.8045), checkpoint saved
    step 0/4  loss=0.2665  elapsed=0s
  ep2 (1s)  train_loss=0.2354 val_loss=0.2378 val_map3=0.8457 val_acc=0.7750 val_f1=0.7741
    -> new best (val_map3=0.8457), checkpoint saved
    step 0/4  loss=0.2016  elapsed=0s
  ep3 (1s)  train_loss=0.1783 val_loss=0.2273 val_map3=0.8512 val_acc=0.7820 val_f1=0.7806
    -> new best (val_map3=0.8512), checkpoint saved


lstm/best_val_map3,▁
lstm/epoch,▁▃▆█
lstm/fold,▁▁▁▁
lstm/train_loss,█▅▂▁
lstm/val_acc,▁▅██
lstm/val_f1,▁▅██
lstm/val_loss,█▃▁▁
lstm/val_map3,▁▅██
lstm/best_val_map3,0.85117
lstm/epoch,3
lstm/fold,0



[LSTM] === fold 1 (train=1000, val=1000) ===
    step 0/4  loss=0.4961  elapsed=0s
  ep0 (1s)  train_loss=0.4610 val_loss=0.4018 val_map3=0.7742 val_acc=0.6540 val_f1=0.6497
    -> new best (val_map3=0.7742), checkpoint saved
    step 0/4  loss=0.3580  elapsed=0s
  ep1 (1s)  train_loss=0.2968 val_loss=0.2625 val_map3=0.8097 val_acc=0.6980 val_f1=0.6963
    -> new best (val_map3=0.8097), checkpoint saved
    step 0/4  loss=0.2060  elapsed=0s
  ep2 (1s)  train_loss=0.1876 val_loss=0.2114 val_map3=0.8260 val_acc=0.7310 val_f1=0.7285
    -> new best (val_map3=0.8260), checkpoint saved
    step 0/4  loss=0.1613  elapsed=0s
  ep3 (1s)  train_loss=0.1532 val_loss=0.2034 val_map3=0.8282 val_acc=0.7350 val_f1=0.7322
    -> new best (val_map3=0.8282), checkpoint saved


lstm/best_val_map3,▁
lstm/epoch,▁▃▆█
lstm/fold,▁▁▁▁
lstm/train_loss,█▄▂▁
lstm/val_acc,▁▅██
lstm/val_f1,▁▅██
lstm/val_loss,█▃▁▁
lstm/val_map3,▁▆██
lstm/best_val_map3,0.82817
lstm/epoch,3
lstm/fold,1


...


Uploading: 100%|██████████| 15.7M/15.7M [00:00<00:00, 17.2MB/s]

Upload successful: /kaggle/working/artifacts/lstm/best_lstm_fold0.pth (15MB)
Starting upload for file /kaggle/working/artifacts/lstm/best_lstm_fold1.pth



Uploading: 100%|██████████| 15.7M/15.7M [00:01<00:00, 14.5MB/s]

Upload successful: /kaggle/working/artifacts/lstm/best_lstm_fold1.pth (15MB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/kamal134134/genai_project/pyTorch/lstm_1
[hub] done. Inference notebooks can now kagglehub.model_download('kamal134134/genai_project/pyTorch/lstm_1').

Training complete. test_lstm shape=(500, 5), oof_lstm shape=(2000, 5)


## DOWNLOADING MODEL FROM KAGGLEHUB

In [1]:
import kagglehub


path = kagglehub.model_download("kamal134134/genai_project/pyTorch/lstm_1")

print("Path to model files:", path)

Path to model files: /kaggle/input/models/kamal134134/genai_project/pytorch/lstm_1/1


## LSTM SUBMISSION

In [7]:
OPTS = ['A', 'B', 'C', 'D', 'E']

test_deb = np.load('/kaggle/input/models/kamal134134/genai_project/pytorch/lstm_1/1/test_lstm.npy')

order = (-test_deb).argsort(axis=1)
preds = [' '.join(OPTS[k] for k in order[i, :3]) for i in range(len(test))]

sub = pd.DataFrame({'ID': test['id'].values, 'Prediction': preds})
sub.to_csv('submission_lstm.csv', index=False)
print("✓ Submission file for LSTM: submission_lstm.csv")

✓ Submission file for LSTM: submission_lstm.csv
